In [2]:
import os
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Config Files - Copy"
OUTPUT_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "7.1-YML_List_3nd_attempt_ShallowC.csv")

# === TEST CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'Firebase_Full': ['gcloud firebase test android run'],
    'Firebase_Compact': ['Firebase-Test-Lab-Action'],
    'Appcenter': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'Browserstack': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices', 'ManagedVirtualDevice', 'managedDevices'],
    'Unit_Test': [
        'gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
        'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'
    ]
}

# === CI-specific instrumentation test keywords ===
ci_instrumentation_keywords = {
    "Travis CI": [
        "connectedCheck", "connectedAndroidTest", "adb shell am instrument",
        "adb wait-for-device", "emulator -avd", "android-wait-for-emulator"
    ],
    "CircleCI": [
        "connectedCheck", "connectedAndroidTest", "circleci/android:", "adb shell",
        "emulator -avd"
    ],
    "GitLab CI": [
        "connectedCheck", "connectedAndroidTest", "adb shell am instrument",
        "emulator -avd", "instrumentation_test"
    ],
    "Bitrise": [
        "connectedCheck", "Start Android Emulator", "android-test",
        "adb shell am instrument"
    ]
}

# === Detect CI platform from path or content ===
def detect_ci_platform(file_path, yaml_text):
    text = yaml_text.lower()
    path = file_path.lower()
    filename = os.path.basename(path)

    if re.search(r'(^|[\\/])\.github([\\/]|$)', path) or 'github_actions' in filename:
        return "GitHub"
    if '.gitlab' in path or 'gitlab-ci' in filename:
        return "GitLab CI"
    if '.travis' in path or 'travis' in filename:
        return "Travis CI"
    if '.circleci' in path or 'circleci' in filename:
        return "CircleCI"
    if '.bitrise' in path or 'bitrise' in filename:
        return "Bitrise"
    if 'uses: actions/' in text:
        return "GitHub"
    if 'travis' in text:
        return "Travis CI"
    if 'circleci' in text:
        return "CircleCI"
    if 'bitrise' in text:
        return "Bitrise"
    if 'gitlab-ci' in text:
        return "GitLab CI"

    return "Other"

# === Detect test types from content ===
def detect_testing_types(yaml_text, ci_platform):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        # Skip generic emulator_manual if CI-specific logic is already applied
        if label == 'emulator_manual' and ci_platform in ci_instrumentation_keywords:
            continue
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)


    # CI-specific instrumentation test detection
    if ci_platform in ci_instrumentation_keywords:
        for instr_kw in ci_instrumentation_keywords[ci_platform]:
            if instr_kw.lower() in uncommented_text:
                found.add(f"{ci_platform}_emulator_manual")
                break

    return found

# === Process all YAML files ===
results = []

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            username = parts[0].lower() if len(parts) > 1 else 'unknown'
            project_name = parts[1].lower() if len(parts) > 2 else parts[0].lower()

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    raw = f.read().replace('\t', ' ')
                    ci_platform = detect_ci_platform(file_path, raw)
                    test_types = detect_testing_types(raw, ci_platform)

                    is_unit_test = 'Unit_Test' in test_types
                    instr_test_types = [
                        t for t in test_types if t != 'Unit_Test'
                    ]

                    # Avoid overlap: drop *_emulator_manual if hosted types exist
                    manual_types = {t for t in instr_test_types if t.endswith("_emulator_manual")}
                    hosted_types = {
                        t for t in instr_test_types
                        if "_emulator_" in t and not t.endswith("_emulator_manual")
                    }
                    if manual_types and hosted_types:
                        instr_test_types = list(hosted_types)

                    test_type_str = ', '.join(sorted(instr_test_types))
                    is_instr = bool(instr_test_types)

                    results.append({
                        'username': username,
                        'project': project_name,
                        'full_name': f"{username}.{project_name}",
                        'ci_platform': ci_platform,
                        'test_type': test_type_str,
                        'unit_test': is_unit_test,
                        'instrumentation_test': is_instr
                    })

            except Exception:
                results.append({
                    'project': project_name,
                    'ci_platform': 'Error',
                    'test_type': '',
                    'unit_test': False,
                    'instrumentation_test': False
                })

# === EXPORT TO CSV ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ 7.1-YML_List.csv generated at: {OUTPUT_CSV}")


✅ 7.1-YML_List.csv generated at: F:\Android_Mobile_App\AndroidProject_3rd\6.2-Shallow_Clone\Analysis Output\7.1-YML_List_3nd_attempt_ShallowC.csv
